In [1]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
import datasets
from datasets import load_dataset

dataset = load_dataset("vlinhd11/viVoice-v1-p1", split="train")

/home/pham/miniconda3/envs/f5v/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 120000/120000 [00:54<00:00, 2221.79 examples/s]


In [2]:
def compute_duration(batch):
    return {
        "duration": [len(a["array"]) / a["sampling_rate"] for a in batch["audio"]]
    }

# Tính nhanh hơn bằng cách dùng batching
dataset_with_duration = dataset.map(compute_duration, batched=True, num_proc=100)

# Tính tổng
total_duration = sum(dataset_with_duration["duration"])
total_duration/3600

Map (num_proc=100):   0%|          | 0/120000 [00:06<?, ? examples/s]


TimeoutError: 

In [3]:
small_dataset = dataset.select(range(100))

In [13]:
import os
import soundfile as sf
import pandas as pd
from multiprocessing import Pool, cpu_count

def process_example(args):
    i, example, output_dir, normalize_text = args
    file_id = f"audio_{i+1}"
    audio_path = os.path.join(output_dir, "wavs", f"{file_id}.wav")

    # Ghi audio
    sf.write(audio_path, example['audio']['array'], example['audio']['sampling_rate'])

    # Xử lý text
    original_text = example['text']
    new_text = original_text.lower() if normalize_text else original_text
    
    if normalize_text:
        return ["wavs/"+file_id+".wav", new_text]
    else:
        return ["wavs/"+file_id+".wav", original_text]

def convert_to_ljspeech_format(dataset, output_dir="ljspeech", normalize_text=False):
    os.makedirs(os.path.join(output_dir, "wavs"), exist_ok=True)

    # Chuẩn bị input cho worker
    args = [(i, ex, output_dir, normalize_text) for i, ex in enumerate(dataset)]

    # Dùng multiprocessing
    with Pool(processes=cpu_count()) as pool:
        metadata = pool.map(process_example, args)

    # Ghi metadata
    # df = pd.DataFrame(metadata, columns=["file_id", "transcription", "normalized_transcription"])
    if normalize_text:
        df = pd.DataFrame(metadata, columns=["file_id", "normalized_transcription"])
    else:
        df = pd.DataFrame(metadata, columns=["file_id", "transcription"])
    df.to_csv(os.path.join(output_dir, "metadata.csv"), sep="|", index=False, header=False)

    print(f"LJSpeech format data saved to: {output_dir}")
demo = True
if demo:
    convert_to_ljspeech_format(small_dataset, output_dir="../data/vivoice_p1_100_sample", normalize_text=True)
else:
    convert_to_ljspeech_format(dataset, output_dir="../vivoice_p1", normalize_text=True)


LJSpeech format data saved to: ../data/vivoice_p1_100_sample
